# Exploratory Data Analysis

Perkiraan waktu: **30** menit

Exploratory Data Analysis (EDA) adalah proses awal untuk mengenali data memakai statistik ringkas dan grafik. Tujuannya menemukan pola, mendeteksi anomali, menguji dugaan, dan memeriksa asumsi sebelum analisis lanjutan. Notebook ini memakai data geografis harga bahan bakar di Kanada sebagai contoh.


## Tujuan

Setelah menyelesaikan latihan ini, kamu dapat:

- Melakukan data wrangling
- Melakukan penyaringan data
- Membuat grafik dengan `plotly.express`
- Menyusun peta choropleth


## Persiapan

Pustaka yang dipakai:

- `pandas` untuk mengelola tabel data.
- `plotly.express` untuk visualisasi.
- `json` dan `urllib` untuk membaca berkas GeoJSON.


In [ ]:
# Imports
import json
import urllib.request
import warnings

import pandas as pd
import plotly.express as px

warnings.filterwarnings("ignore")


## Membaca dan memahami data

Data yang dipakai adalah *Monthly average retail prices for gasoline and fuel oil, by geography* dari Statistics Canada. Isinya harga rata-rata bulanan bahan bakar (sen per liter) di kota-kota besar Kanada, mulai 1979 sampai tahun terakhir pencatatan.

Berkas `canada_provinces.geojson` menyimpan batas wilayah provinsi Kanada dan dipakai nanti untuk peta choropleth.


In [ ]:
# Muat data harga bahan bakar langsung dari sumber publik
fuel_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/18100001.csv"
fuel_data = pd.read_csv(fuel_url, encoding="utf-8-sig")
fuel_data.head()


Jumlah baris dan kolom diperiksa dengan `shape`, lalu tipe datanya dengan `info()`.


In [ ]:
print(f"Jumlah baris dan kolom: {fuel_data.shape}")
fuel_data.info()


In [ ]:
fuel_data.columns


Nilai kosong dihitung per kolom untuk melihat kelengkapan data.


In [ ]:
fuel_data.isnull().sum()


## Data wrangling

### Memilih dan mengganti nama kolom

Hanya empat kolom yang relevan untuk analisis, dan dua di antaranya diberi nama lebih ringkas.


In [ ]:
geo_data = (
    fuel_data[["REF_DATE", "GEO", "Type of fuel", "VALUE"]]
    .rename(columns={"REF_DATE": "DATE", "Type of fuel": "TYPE"})
)
geo_data.head()


### Memecah kolom

`str.split()` memecah teks berdasarkan koma. Dengan `n=1` hanya koma pertama yang dipakai, dan `expand=True` mengembalikan hasilnya sebagai DataFrame. Kolom `GEO` dipecah menjadi `City` dan `Province`.


In [ ]:
geo_data[["City", "Province"]] = geo_data["GEO"].str.split(",", n=1, expand=True)
geo_data[["GEO", "City", "Province"]].head()


### Mengubah ke format datetime

`REF_DATE` awalnya bertipe teks dengan pola bulan-tahun, misalnya `Jan-79`. Agar bisa disaring berdasarkan bulan atau tahun, kolom ini diubah memakai `to_datetime()` dengan format `%b-%y`. Setelah itu nama bulan dan tahun diambil lewat `.dt`.


In [ ]:
geo_data["DATE"] = pd.to_datetime(geo_data["DATE"], format="%b-%y")
geo_data["Month"] = geo_data["DATE"].dt.month_name().str.slice(stop=3)
geo_data["Year"] = geo_data["DATE"].dt.year
geo_data[["DATE", "Month", "Year"]].head()


Statistik kolom `VALUE` dilihat dengan `describe()`.


In [ ]:
geo_data["VALUE"].describe()


Isi kolom kategorikal `GEO` diperiksa memakai `unique().tolist()`.


In [ ]:
geo_data["GEO"].unique().tolist()


## Latihan 1

Tampilkan seluruh kategori pada kolom `TYPE`.


In [ ]:
geo_data["TYPE"].unique().tolist()


Terdapat tujuh jenis bahan bakar, mulai dari bensin reguler, premium, diesel, sampai bahan bakar pemanas rumah tangga.

## Penyaringan data

### Penyaringan dengan operator logika

Operator logika diterapkan pada nilai kolom untuk memilih baris. Tanda `==` memilih nilai yang sama, `!=` mengecualikan nilai tertentu, sedangkan `<`, `>`, `<=`, dan `>=` dipakai untuk nilai numerik.

Contoh pertama mengambil seluruh data Calgary, Alberta.


In [ ]:
calgary_rows = geo_data[geo_data["GEO"] == "Calgary, Alberta"]
calgary_rows


Berikutnya data tahun 2000.


In [ ]:
year_2000 = geo_data[geo_data["Year"] == 2000]
year_2000


### Penyaringan dengan beberapa kondisi

Beberapa kondisi digabung memakai `|` (atau) dan `&` (dan). Contoh berikut memilih data Toronto dan Edmonton sekaligus.


In [ ]:
two_cities = geo_data[
    (geo_data["GEO"] == "Toronto, Ontario")
    | (geo_data["GEO"] == "Edmonton, Alberta")
]
two_cities


Cara lain untuk memilih beberapa nilai sekaligus adalah `isin()`.


In [ ]:
selected_cities = ["Calgary", "Toronto", "Edmonton"]
city_subset = geo_data[geo_data["City"].isin(selected_cities)]
city_subset


## Latihan 2a

Pilih data harga *household heating fuel* di Vancouver pada tahun 1990.


In [ ]:
vancouver_1990 = geo_data[
    (geo_data["Year"] == 1990)
    & (geo_data["TYPE"] == "Household heating fuel")
    & (geo_data["City"] == "Vancouver")
]
vancouver_1990


## Latihan 2b

Pilih data harga *household heating fuel* di Vancouver pada tahun 1979 dan 2021.


In [ ]:
vancouver_two_years = geo_data[
    geo_data["Year"].isin([1979, 2021])
    & (geo_data["TYPE"] == "Household heating fuel")
    & (geo_data["City"] == "Vancouver")
]
vancouver_two_years


Hasilnya hanya memuat data tahun 2021. Untuk bahan bakar pemanas rumah tangga di Vancouver, data tahun 1979 memang tidak tersedia. Karena itu operator `|` atau `isin()` lebih tepat dibanding `&`, sebab `&` akan menuntut kedua tahun terisi dan menghasilkan tabel kosong.

### Penyaringan dengan `groupby()`

`groupby()` memecah data berdasarkan kategori. Pemanggilan paling sederhana cukup memakai nama kolom. Atribut `ngroups` menunjukkan jumlah kelompok pada kolom `GEO`.


In [ ]:
geo_groups = geo_data.groupby("GEO")
geo_groups.ngroups


Fungsi agregasi seperti `mean`, `median`, `min`, `max`, dan `count` diterapkan pada setiap kelompok, lalu hasilnya digabung kembali. Contoh berikut menghitung rata-rata harga per tahun.


In [ ]:
yearly_average = geo_data.groupby("Year")["VALUE"].mean()
yearly_average


## Latihan 3a

Gunakan `groupby()` untuk menghitung harga tertinggi pada setiap bulan.


In [ ]:
monthly_max = geo_data.groupby("Month")["VALUE"].max()
monthly_max


## Latihan 3b

Gunakan `groupby()` untuk menghitung median harga pada setiap tahun dan setiap kota.


In [ ]:
yearly_city_median = (
    geo_data.groupby(["Year", "City"])["VALUE"]
    .median()
    .reset_index(name="Value")
    .round(2)
)
yearly_city_median


`reset_index(name=...)` mengubah hasil groupby menjadi tabel biasa dengan kolom nilai yang diberi nama, lalu `round(2)` membulatkan nilainya menjadi dua angka desimal.

## Visualisasi dengan plotly.express

`plotly.express` (biasa diimpor sebagai `px`) membuat satu figure utuh dalam satu pemanggilan fungsi. Grafik pertama menampilkan tren harga rata-rata tahunan untuk setiap kota.


In [ ]:
price_per_city = (
    geo_data.groupby(["Year", "GEO"])["VALUE"]
    .mean()
    .reset_index(name="Value")
    .round(2)
)

figure = px.line(
    price_per_city,
    x="Year",
    y="Value",
    color="GEO",
    color_discrete_sequence=px.colors.qualitative.Light24,
)
figure.update_traces(mode="markers+lines")
figure.update_layout(
    title="Gasoline Price Trend per City",
    xaxis_title="Year",
    yaxis_title="Annual Average Price, Cents per Litre",
)
figure.show()


Grafik kedua memperlihatkan harga bulanan rata-rata di Toronto sepanjang 2021.


In [ ]:
toronto_2021 = geo_data[
    (geo_data["Year"] == 2021) & (geo_data["GEO"] == "Toronto, Ontario")
]
toronto_monthly = (
    toronto_2021.groupby("Month")["VALUE"]
    .mean()
    .reset_index()
    .sort_values(by="VALUE")
)

figure = px.line(toronto_monthly, x="Month", y="VALUE")
figure.update_traces(mode="markers+lines")
figure.update_layout(
    title="Toronto Average Monthly Gasoline Price in 2021",
    xaxis_title="Month",
    yaxis_title="Monthly Price, Cents per Litre",
)
figure.show()


## Latihan 4

Buat grafik tren harga rata-rata tahunan untuk setiap jenis bahan bakar memakai `plotly.express`.


In [ ]:
price_per_type = (
    geo_data.groupby(["Year", "TYPE"])["VALUE"]
    .mean()
    .reset_index(name="Value")
    .round(2)
)

figure = px.line(
    price_per_type,
    x="Year",
    y="Value",
    color="TYPE",
    color_discrete_sequence=px.colors.qualitative.Light24,
)
figure.update_traces(mode="markers+lines")
figure.update_layout(
    title="Fuel Type Price Trend",
    xaxis_title="Year",
    yaxis_title="Annual Average Price, Cents per Litre",
)
figure.show()


Tren waktu juga bisa ditampilkan sebagai animasi. Setiap frame mewakili satu tahun, sehingga pergeseran harga antar kota terlihat langsung.


In [ ]:
price_per_city_simple = (
    geo_data.groupby(["Year", "City"])["VALUE"]
    .mean()
    .reset_index(name="Value")
    .round(2)
)
price_per_city_simple.head()


In [ ]:
figure = px.bar(
    price_per_city_simple, x="City", y="Value", animation_frame="Year"
)
figure.update_layout(
    title="Time Lapse of Average Price of Gasoline, by Province",
    xaxis_title="Year",
    yaxis_title="Average Price of Gasoline, Cents per Litre",
)
figure.show()


Sebaran harga antar provinsi bisa ditampilkan sebagai peta. Data tahun 2021 dipakai sebagai contoh. Pertama, data tahun tersebut disaring.


In [ ]:
latest_year = geo_data[geo_data["Year"] == 2021]
latest_year.head()


Harga dirata-ratakan per provinsi. Setiap provinsi kemudian diberi nomor `ProvinceID` yang sesuai dengan `cartodb_id` pada berkas GeoJSON.


In [ ]:
province_price = (
    latest_year.groupby("Province")["VALUE"]
    .mean()
    .reset_index(name="Average Gasoline Price")
    .round(2)
)
province_price["Province"] = province_price["Province"].str.strip()

province_ids = {
    "Newfoundland and Labrador": 5,
    "Prince Edward Island": 8,
    "Nova Scotia": 2,
    "New Brunswick": 7,
    "Quebec": 1,
    "Ontario": 11,
    "Ontario part, Ontario/Quebec": 12,
    "Manitoba": 10,
    "Saskatchewan": 3,
    "Alberta": 4,
    "British Columbia": 6,
    "Yukon": 9,
    "Northwest Territories": 13,
}
province_price["ProvinceID"] = province_price["Province"].map(province_ids)
province_price


Berkas GeoJSON diunduh dari penyimpanan publik, lalu isinya diubah menjadi objek Python dengan `json.loads()`.


In [ ]:
geojson_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML0232EN-SkillsNetwork/asset/canada_provinces.geojson"
with urllib.request.urlopen(geojson_url) as response:
    canada_map = json.loads(response.read().decode("utf-8"))

len(canada_map["features"])


`ProvinceID` pada tabel dihubungkan dengan `properties.cartodb_id` pada GeoJSON. Warna wilayah mengikuti harga rata-rata bahan bakar, dan tampilan peta dibatasi hanya pada wilayah Kanada.


In [ ]:
figure = px.choropleth(
    province_price,
    locations="ProvinceID",
    geojson=canada_map,
    featureidkey="properties.cartodb_id",
    color="Average Gasoline Price",
    color_continuous_scale=px.colors.diverging.Tropic,
    scope="north america",
    title="<b>Average Gasoline Price</b>",
    hover_name="Province",
    hover_data={"Average Gasoline Price": True, "ProvinceID": False},
    locationmode="geojson-id",
)
figure.update_layout(
    showlegend=True,
    legend_title_text="<b>Average Gasoline Price</b>",
    font={"size": 16, "color": "#808080", "family": "calibri"},
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    legend=dict(orientation="v"),
    geo=dict(bgcolor="rgba(0,0,0,0)", lakecolor="#e0fffe"),
)
figure.update_geos(
    showcountries=False,
    showcoastlines=False,
    showland=False,
    fitbounds="locations",
    subunitcolor="white",
)
figure.show()


## Latihan 5

Coba beberapa skala warna lain agar peta lebih mudah dibaca. Skala sekuensial cocok untuk data kontinu, sedangkan skala divergen berguna bila data punya titik tengah alami.


In [ ]:
figure = px.choropleth(
    province_price,
    locations="ProvinceID",
    geojson=canada_map,
    featureidkey="properties.cartodb_id",
    color="Average Gasoline Price",
    color_continuous_scale=px.colors.sequential.Greens,
    scope="north america",
    title="<b>Average Gasoline Price</b>",
    hover_name="Province",
    hover_data={"Average Gasoline Price": True, "ProvinceID": False},
    locationmode="geojson-id",
)
figure.update_geos(
    showcountries=False,
    showcoastlines=False,
    showland=False,
    fitbounds="locations",
    subunitcolor="white",
)
figure.show()
